In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Understand plans and dags")
    .master("local[*]")
    .getOrCreate()
)

In [4]:
# Disable AQE and Broadcast join

spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")

In [5]:
# Check default parallelism 
spark.sparkContext.defaultParallelism

36

In [7]:
# create dataframe
df1 = spark.range(4, 200, 2)
df2 = spark.range(2, 200, 4)

In [9]:
df2.rdd.getNumPartitions()

36

In [10]:
# re-partition data
df3 = df1.repartition(5)
df4 = df2.repartition(7)

In [11]:
df3.rdd.getNumPartitions()

5

In [12]:
# Join the dataframes
df_joined = df3.join(df4, on='id', how='inner')

In [15]:
# get the sum of ids
df_sum = df_joined.selectExpr("sum(id) as sum_id")
df_sum.show()

+------+
|sum_id|
+------+
|  4998|
+------+



In [17]:
df_sum.explain()

== Physical Plan ==
*(6) HashAggregate(keys=[], functions=[sum(id#2L)])
+- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=170]
   +- *(5) HashAggregate(keys=[], functions=[partial_sum(id#2L)])
      +- *(5) Project [id#2L]
         +- *(5) SortMergeJoin [id#2L], [id#3L], Inner
            :- *(2) Sort [id#2L ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(id#2L, 200), ENSURE_REQUIREMENTS, [plan_id=154]
            :     +- Exchange RoundRobinPartitioning(5), REPARTITION_BY_NUM, [plan_id=153]
            :        +- *(1) Range (4, 200, step=2, splits=36)
            +- *(4) Sort [id#3L ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(id#3L, 200), ENSURE_REQUIREMENTS, [plan_id=161]
                  +- Exchange RoundRobinPartitioning(7), REPARTITION_BY_NUM, [plan_id=160]
                     +- *(3) Range (2, 200, step=4, splits=36)


